# Deep Research Agent trên Google Colab

Notebook chạy lab với **9router (OpenAI-compatible) + Daytona**, sinh 5 báo cáo và lưu từng bộ kết quả vào Drive. Không cần GPU.

1. Chạy ô cài đặt để lấy mã từ repo GitHub. Có thể đổi SOURCE sang upload để dùng colab_bundle.zip.
2. Trong Colab Secrets, bật Notebook access cho LAB_BASE_URL (HTTPS endpoint /v1 truy cập được từ Colab), LAB_MODEL (tên model hỗ trợ tool calling), LAB_API_KEY và DAYTONA_API_KEY. Notebook ưu tiên DAYTONA_LAB_API_KEY nếu có. EXA_API_KEY là tùy chọn.
3. Với 9router chạy trên máy cá nhân, dùng endpoint tunnel HTTPS đang có trong dashboard; localhost trên Colab trỏ tới máy Colab. Giữ máy chạy 9router và tunnel hoạt động.
4. Chạy các ô lần lượt. Cấu hình lấy từ Secrets, không hỏi nhập khóa trong khi chạy. Nếu thiếu cấu hình, ô kiểm tra dừng và chỉ nêu tên biến còn thiếu.
5. Sau khi cấp quyền Drive lần đầu, kết quả được lưu tại MyDrive/DeepResearchLab/reports. Khi runtime ngắt, chạy lại các ô: chỉ báo cáo đầy đủ, hợp lệ mới được bỏ qua; chủ đề dở dang chạy lại.

Khóa chỉ ở host, không đưa vào sandbox, ZIP hay Git. Kiểm tra tự động không thay thế chấm chất lượng và kiểm tra bằng chứng của giảng viên.

Model mặc định cho kết nối 9router hiện có: `ag/gemini-3.1-pro-high`. Trong ô cấu hình, đặt `MODEL_OVERRIDE` rỗng để dùng `LAB_MODEL` từ Secrets hoặc đổi tên model theo provider của bạn.


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile

SOURCE = "github" # @param ["upload", "github"]
REPO_URL = "https://github.com/Neon310304/K4-Day23-AdvanceDeepAgents-Labs.git"
PROJECT = Path("/content/deep-research-agent")
if SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    archive_names = [name for name in uploaded if name.endswith('.zip')]
    if len(archive_names) != 1:
        raise ValueError('Upload exactly one colab_bundle.zip file')
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_names[0]) as archive:
        for entry in archive.infolist():
            target = (PROJECT / entry.filename).resolve()
            if not target.is_relative_to(PROJECT.resolve()):
                raise ValueError('Unsafe archive path')
            if Path(entry.filename).name == '.env':
                raise ValueError('The source bundle must never contain .env')
        archive.extractall(PROJECT)
else:
    if not (PROJECT / '.git').exists():
        import tempfile, shutil
        # A previous ZIP install can leave a nonempty project without .git.
        # Install public source via a temporary clone and retain saved reports.
        with tempfile.TemporaryDirectory(prefix='deep-research-install-') as temporary:
            checkout = Path(temporary) / 'source'
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(checkout)], check=True)
            shutil.copytree(checkout, PROJECT, dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns('.git', 'reports', '.env'))
    else:
        print('Using existing checkout. Upload a new bundle if you need updated code.')
if not (PROJECT / 'requirements.txt').is_file():
    raise RuntimeError('The bundle must have requirements.txt at its root')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
print('Installed project at', PROJECT)


In [ ]:
import hashlib, sys, os
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
expected={
'model.py':'37e837b13acea6a4be598f8057e623738d24a0d356562a7b386e961b24a644a0',
'sandbox.py':'b10efee205cdcd245eb3b64b9efbc755418fa35556797e039b6b7b40ed2d3db4',
'self_check.py':'5d98dc31b05c0995557955c12b85a48291d00d58049dd2fb5616d087ee88e45f',
'finalize_citations.py':'4961e18c3fe6e575ee5db474ec20f6cf57f96f8a82a38a65514a4872690e3a76'}
for name,digest in expected.items():
    original_source=(PROJECT/name).read_bytes().replace(b'\r\n',b'\n')
    if hashlib.sha256(original_source).hexdigest()!=digest:
        raise RuntimeError('Protected source changed: '+name)
os.environ['LAB_PROGRESS']='1'
print('PROTECTED_FILES: 4 match original source (Git line endings normalized)')


In [ ]:
import os
from urllib.parse import urlsplit
from google.colab import userdata

# 9router uses the OpenAI-compatible configuration of the provided model.py.
# You can also use another compatible provider by changing these Secrets.
def configured(name, *alternatives, required=True):
    for candidate in (name, *alternatives):
        try:
            value = userdata.get(candidate)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            value = os.environ.get(candidate, '')
        value = (value or '').strip()
        if value:
            os.environ[name] = value
            return value
    if required:
        raise RuntimeError('Missing Colab Secret or environment variable: ' + name)
    return ''

for alias in ('OPENAI_ENDPOINT', 'OPENAI_KEY', 'OPENAI_DEPLOYMENT_MODEL'):
    os.environ.pop(alias, None)
base_url = configured('LAB_BASE_URL')
parsed = urlsplit(base_url)
if parsed.scheme != 'https' or not parsed.hostname or parsed.query or parsed.username or parsed.password:
    raise ValueError('LAB_BASE_URL must be HTTPS without embedded credentials or query parameters')
MODEL_OVERRIDE = "ag/gemini-3.1-pro-high" # @param {type:"string"}
if MODEL_OVERRIDE.strip():
    os.environ['LAB_MODEL']=MODEL_OVERRIDE.strip()
else:
    configured('LAB_MODEL')
configured('LAB_API_KEY')
# Prefer the dedicated sandbox key without changing the user's other secret.
try:
    dedicated_key = userdata.get('DAYTONA_LAB_API_KEY').strip()
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    dedicated_key = ''
if dedicated_key:
    os.environ['DAYTONA_API_KEY'] = dedicated_key
else:
    configured('DAYTONA_API_KEY')
configured('EXA_API_KEY', required=False)
os.environ['SANDBOX'] = 'daytona'
os.environ['LAB_PROGRESS'] = '1'
print('Model:', os.environ['LAB_MODEL'], '| Sandbox: Daytona | Secrets loaded (values hidden)')

# Public paper HTML can be retrieved on the host without Exa free-tier limits.
os.environ['LAB_DIRECT_PRIMARY_FETCH']='1'


In [ ]:
# Optional persistence: valid reports can survive Colab runtime resets.
USE_DRIVE = True # @param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    DRIVE_REPORTS = Path('/content/drive/MyDrive/DeepResearchLab/reports')
    DRIVE_REPORTS.mkdir(parents=True, exist_ok=True)
    (PROJECT / 'reports').mkdir(exist_ok=True)
    for saved in DRIVE_REPORTS.iterdir():
        if saved.is_file() and (saved.name.endswith('.md') or saved.name.endswith('.json')):
            shutil.copy2(saved, PROJECT / 'reports' / saved.name)
else:
    DRIVE_REPORTS = None


In [ ]:
# Offline regression tests and real source-tool checks.
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                        cwd=PROJECT, capture_output=True, text=True)
print(result.stdout); print(result.stderr); result.check_returncode()

import re
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
import research
existing_topics=re.findall(r'^\d+\. (.+)$',(PROJECT/'topics.md').read_text(encoding='utf-8'),re.M)
if existing_topics and all(research._complete_existing(t) for t in existing_topics):
    print('VERIFIED_EXISTING:',len(existing_topics),'reports; skipping network/model smoke tests')
else:
    result = subprocess.run([sys.executable, '-u', 'tools.py'], cwd=PROJECT,
                            capture_output=True, text=True)
    print(result.stdout); print(result.stderr); result.check_returncode()
    if str(PROJECT) not in sys.path:
        sys.path.insert(0, str(PROJECT))
    from langchain_core.tools import tool
    from model import make_model
    @tool
    def lab_connectivity_probe() -> str:
        """Return a harmless connectivity probe; no external side effects."""
        return 'ok'
    reply = make_model().bind_tools([lab_connectivity_probe], tool_choice='lab_connectivity_probe').invoke(
        'Call lab_connectivity_probe exactly once, with no arguments.')
    if not reply.tool_calls or reply.tool_calls[0]['name'] != 'lab_connectivity_probe':
        raise RuntimeError('Selected model did not return the requested tool call')
    print('LLM_TOOL_CALLING_OK: True')
    

In [ ]:
import hashlib, sys, os
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
expected={
'model.py':'37e837b13acea6a4be598f8057e623738d24a0d356562a7b386e961b24a644a0',
'sandbox.py':'b10efee205cdcd245eb3b64b9efbc755418fa35556797e039b6b7b40ed2d3db4',
'self_check.py':'5d98dc31b05c0995557955c12b85a48291d00d58049dd2fb5616d087ee88e45f',
'finalize_citations.py':'4961e18c3fe6e575ee5db474ec20f6cf57f96f8a82a38a65514a4872690e3a76'}
for name,digest in expected.items():
    original_source=(PROJECT/name).read_bytes().replace(b'\r\n',b'\n')
    if hashlib.sha256(original_source).hexdigest()!=digest:
        raise RuntimeError('Protected source changed: '+name)
os.environ['LAB_PROGRESS']='1'
print('PROTECTED_FILES: 4 match original source (Git line endings normalized)')

import importlib, re
importlib.invalidate_caches()
for name in ('validate_report','tools','workflow','agents','research'):
    if name in sys.modules: importlib.reload(sys.modules[name])
import research
assert research.PIPELINE_VERSION == 'parallel-v3'
assert research.REPAIR_RECURSION_LIMIT == 1000
topics=re.findall(r'^\d+\. (.+)$',(PROJECT/'topics.md').read_text(encoding='utf-8'),re.M)
print('PIPELINE:',research.PIPELINE_VERSION,'| RECURSION:',research.RECURSION_LIMIT,research.REPAIR_RECURSION_LIMIT)
print('RESUMABLE_REPORTS:',sum(research._complete_existing(t) for t in topics),'/',len(topics))


In [ ]:
# Run each topic once, skip verified completed outputs and persist after each run.
import re, shutil, importlib
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
importlib.invalidate_caches()
# Reload updated project code after a bundle upload in the same runtime.
for module_name in ('validate_report', 'tools', 'workflow', 'agents', 'research'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
import research
if (getattr(research, 'PIPELINE_VERSION', None) != 'parallel-v3'
        or getattr(research, 'REPAIR_RECURSION_LIMIT', 0) < 1000):
    raise RuntimeError('Upload the latest colab_bundle.zip before running this cell')
print('Report pipeline:', research.PIPELINE_VERSION)
print('Repair graph step limit:', research.REPAIR_RECURSION_LIMIT)
topics = re.findall(r'^\d+\. (.+)$', (PROJECT / 'topics.md').read_text(encoding='utf-8'), re.M)
for topic in topics:
    if research._complete_existing(topic):
        print('Skipping verified report:', topic)
    else:
        code = research.main(topic)
        if code:
            raise RuntimeError('Research failed. Fix configuration/evidence issue, then rerun this cell: ' + topic)
    if DRIVE_REPORTS is not None:
        slug = research.slugify(topic)
        for suffix in ('.md', '.sources.json', '.meta.json'):
            shutil.copy2(PROJECT / 'reports' / (slug + suffix), DRIVE_REPORTS / (slug + suffix))
subprocess.run([sys.executable, 'self_check.py', '--no-git'], cwd=PROJECT, check=True)


In [ ]:
import hashlib, json
verified = 0
for topic in topics:
    if not research._complete_existing(topic):
        raise RuntimeError('Incomplete report: ' + topic)
    slug = research.slugify(topic)
    meta = json.loads((PROJECT/'reports'/(slug+'.meta.json')).read_text(encoding='utf-8'))
    for suffix in ('.md','.sources.json','.meta.json'):
        path = PROJECT/'reports'/(slug+suffix)
        if DRIVE_REPORTS is not None and path.read_bytes() != (DRIVE_REPORTS/path.name).read_bytes():
            raise RuntimeError('Drive copy differs: '+path.name)
    verified += 1
    print(f"PASS | {topic} | sources={meta['n_sources']} | tasks={meta['subagent_calls']} | families={','.join(meta['source_families'])}")
print(f'FINAL: {verified}/{len(topics)} reports verified; {verified*3} artifacts; Drive matches' if DRIVE_REPORTS is not None else f'FINAL: {verified}/{len(topics)} reports verified; {verified*3} artifacts')
print('Report/source content is preserved from sandbox downloads. Scientific evidence review: SUBMISSION_REVIEW.md in GitHub.')


In [ ]:
# Export verified report artifacts without including host secrets.
import hashlib, shutil, zipfile
EXPORT = Path('/content/deep-research-reports.zip')
manifest = []
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as archive:
    for topic in topics:
        if not research._complete_existing(topic):
            raise RuntimeError('Incomplete report: ' + topic)
        slug = research.slugify(topic)
        for suffix in ('.md', '.sources.json', '.meta.json'):
            path = PROJECT / 'reports' / (slug + suffix)
            payload = path.read_bytes()
            if DRIVE_REPORTS is not None and payload != (DRIVE_REPORTS/path.name).read_bytes():
                raise RuntimeError('Drive mismatch: ' + path.name)
            archive.writestr('reports/' + path.name, payload)
            manifest.append(hashlib.sha256(payload).hexdigest() + '  reports/' + path.name)
manifest_path = PROJECT / 'REPORTS.sha256'
manifest_path.write_text('\n'.join(sorted(manifest, key=lambda line: line.split('  ', 1)[1])) + '\n', encoding='utf-8')
if DRIVE_REPORTS is not None:
    shutil.copy2(EXPORT, DRIVE_REPORTS.parent/EXPORT.name)
    shutil.copy2(manifest_path, DRIVE_REPORTS.parent/manifest_path.name)
print('EXPORT_OK:', len(manifest), 'artifacts;', EXPORT.name, 'saved to Drive' if DRIVE_REPORTS is not None else str(EXPORT))
DOWNLOAD_ZIP = False # @param {type:"boolean"}
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(EXPORT))


## Bài nộp hoàn thiện

Đã chạy đủ 5 chủ đề; mỗi chủ đề có báo cáo, sources.json và meta.json. Kết quả được lưu ở MyDrive/DeepResearchLab/reports, ZIP và REPORTS.sha256 ở thư mục DeepResearchLab.

[Repo GitHub công khai](https://github.com/Neon310304/K4-Day23-AdvanceDeepAgents-Labs) · [Bảng tự đánh giá theo RUBRIC](https://github.com/Neon310304/K4-Day23-AdvanceDeepAgents-Labs/blob/main/SUBMISSION_REVIEW.md)

81 kiểm thử và self_check đã đạt. Báo cáo/nguồn là đúng bytes tải từ sandbox; bốn tệp cấp sẵn giữ nguyên. Điểm tự chấm là đánh giá nội bộ, điểm chính thức do giảng viên quyết định.

Khi chạy lại, các báo cáo hợp lệ được xác minh và bỏ qua; chủ đề dở dang chạy lại. Khóa API chỉ được đọc từ Secrets, không hỏi nhập trong vòng chạy.
